In [0]:
%run ../common/common_logging

In [0]:
%run ../common/common_http_client

In [0]:
%run ../common/common_hashing

In [0]:
%run ../common/common_config_loader

In [0]:
%run ./ingest_manifest_writer

In [0]:
# Get configuration
config = get_config()
logger = get_logger(__name__)

ADZUNA_API = "https://api.adzuna.com/v1/api/jobs/us/search/1"
SOURCE_NAME = "adzuna"
BRONZE_JOB_SNAPSHOT = config.get_bronze_table("bronze_job_snapshot")

In [0]:
import re
import time
import random

def strip_html(text):
    if not text:
        return ""
    clean = re.sub(r'<script.*?>.*?</script>', '', text, flags=re.DOTALL)
    clean = re.sub(r'<style.*?>.*?</style>', '', clean, flags=re.DOTALL)
    clean = re.sub(r'<[^>]*>', ' ', clean)
    clean = re.sub(r'\s+', ' ', clean)
    return clean.strip()

def generate_mock_adzuna():
    """Generate mock retail and hospitality jobs for fallback / testing"""
    mock_titles = [
        ("Retail Sales Associate", "RETAIL_ASSOC", "Target", "Retail Ops", "Help customers on the sales floor. Cashiering and merchandising skills preferred."),
        ("Store Manager", "RETAIL_MGR", "Walmart", "Retail Ops", "Manage store operations and lead retail associate team. Strong leadership and inventory management skills."),
        ("Line Cook", "HOSP_COOK", "Marriott", "Food & Beverage", "Prepare food on the line. Knowledge of food safety and culinary arts required."),
        ("Bartender", "HOSP_BARTENDER", "Hilton Hotels", "Food & Beverage", "Serve drinks and manage bar inventory. Customer service and mixology experience expected."),
        ("Visual Merchandiser", "RETAIL_MERCHANDISER", "Zara", "Retail Ops", "Create visually appealing displays and visual merchandising plans."),
        ("Registered Nurse", "HEAL_RN", "General Hospital", "Clinical Care", "Provide bedside care and coordinate clinical compliance.")
    ]
    
    jobs = []
    now_epoch = int(time.time())
    for i, (title, role_key, company, category, desc) in enumerate(mock_titles):
        jobs.append({
            "id": f"adzuna-{i+2000}",
            "title": title,
            "description": f"<div class='job-desc'><strong>{title}</strong>: {desc} Includes project management and team collaboration.</div>",
            "company": {"display_name": company},
            "location": {"display_name": "New York, NY"},
            "redirect_url": f"https://www.adzuna.com/details/{i+2000}",
            "created": "2026-06-01T12:00:00Z"
        })
    return jobs

def fetch_adzuna_jobs():
    """Fetch from Adzuna API or fallback to mock data"""
    # Check if we have credentials
    app_id = None
    app_key = None
    try:
        app_id = dbutils.secrets.get(scope="lmip-scope", key="ADZUNA_APP_ID")
        app_key = dbutils.secrets.get(scope="lmip-scope", key="ADZUNA_APP_KEY")
    except:
        pass
        
    if not app_id or not app_key:
        logger.info("No credentials found for Adzuna API - falling back to mock data")
        return generate_mock_adzuna(), None
        
    try:
        headers = {"Accept": "application/json"}
        params = {
            "app_id": app_id,
            "app_key": app_key,
            "results_per_page": 10,
            "what": "retail"
        }
        http_config = HTTPClientConfig(max_retries=3, connect_timeout=10, read_timeout=30)
        client = HTTPClient(base_url=ADZUNA_API, config=http_config)
        data = client.get("", headers=headers, params=params)
        results = data.get("results", [])
        return results, None
    except Exception as e:
        logger.error(f"Failed to fetch from Adzuna API, returning mock data: {e}")
        return generate_mock_adzuna(), None

def extract_adzuna_job_id(job):
    return str(job.get("id", ""))

def validate_adzuna_record(job):
    required = ["id", "title", "description", "redirect_url"]
    for field in required:
        if not job.get(field):
            return False, f"Missing {field}"
    if not job.get("company", {}).get("display_name"):
        return False, "Missing company display name"
    return True, None

def parse_to_common_schema(job):
    raw_desc = job.get("description", "")
    clean_desc = strip_html(raw_desc)
    loc_str = job.get("location", {}).get("display_name", "Unknown")
    
    # Standardized payload for Silver layer processing
    return {
        "company_name": job.get("company", {}).get("display_name", ""),
        "title": job.get("title", ""),
        "description": clean_desc,
        "location": loc_str,
        "remote": False,
        "url": job.get("redirect_url", ""),
        "created_at": int(time.time() * 1000),
        "candidate_required_location": loc_str
    }

In [0]:
def ingest_adzuna():
    batch_id = generate_batch_id()
    run_control_sk = start_pipeline_run("bronze_ingestion_adzuna", SOURCE_NAME, batch_id)
    start_time = time.time()
    
    try:
        jobs, error = fetch_adzuna_jobs()
        if error:
            log_api_response(SOURCE_NAME, batch_id, ADZUNA_API, 500, response_time_ms=0)
            complete_pipeline_run(batch_id, 'FAILED')
            log_audit_pipeline_run(batch_id, "bronze_ingestion_adzuna", 'FAILED', runtime_seconds=time.time()-start_time)
            return False
            
        log_api_response(SOURCE_NAME, batch_id, ADZUNA_API, 200, response_time_ms=100)
        
        # Transform and write to Bronze
        bronze_records = []
        now = datetime.now(timezone.utc)
        
        for job in jobs:
            is_valid, _ = validate_adzuna_record(job)
            if not is_valid: continue
            
            job_id = extract_adzuna_job_id(job)
            common_payload = parse_to_common_schema(job)
            payload_json = json.dumps(common_payload)
            payload_hash = calculate_payload_hash(common_payload)
            snapshot_id = f"{SOURCE_NAME}_{job_id}_{batch_id}"
            
            bronze_records.append({
                'snapshot_id': snapshot_id,
                'source_name': SOURCE_NAME,
                'source_job_id': job_id,
                'batch_id': batch_id,
                'page_number': None,
                'page_size': None,
                'payload_json': payload_json,
                'payload_hash': payload_hash,
                'ingestion_timestamp': now,
                'ingestion_date': now.date(),
                'source_status_code': 200,
                'source_etag': None
            })
            
        if bronze_records:
            bronze_schema = StructType([
                StructField("snapshot_id", StringType(), False),
                StructField("source_name", StringType(), False),
                StructField("source_job_id", StringType(), True),
                StructField("batch_id", StringType(), False),
                StructField("page_number", IntegerType(), True),
                StructField("page_size", IntegerType(), True),
                StructField("payload_json", StringType(), False),
                StructField("payload_hash", StringType(), False),
                StructField("ingestion_timestamp", TimestampType(), False),
                StructField("ingestion_date", DateType(), False),
                StructField("source_status_code", IntegerType(), True),
                StructField("source_etag", StringType(), True)
            ])
            df = spark.createDataFrame(bronze_records, schema=bronze_schema)
            df.write.mode('append').saveAsTable(BRONZE_JOB_SNAPSHOT)
            
        duration = time.time() - start_time
        complete_pipeline_run(batch_id, 'SUCCESS')
        log_audit_pipeline_run(batch_id, "bronze_ingestion_adzuna", 'SUCCESS', rows_read=len(jobs), rows_written=len(bronze_records), runtime_seconds=duration)
        print(f"✓ Ingested {len(bronze_records)} records")
        return True
    except Exception as e:
        complete_pipeline_run(batch_id, 'FAILED')
        log_audit_pipeline_run(batch_id, "bronze_ingestion_adzuna", 'FAILED', runtime_seconds=time.time()-start_time, error_message=str(e))
        raise e

ingest_adzuna()